In [1]:
%%capture
!pip install --upgrade transformers bitsandbytes accelerate peft datasets wandb contractions spacy tqdm

In [2]:
import torch
import wandb
import pandas as pd
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset, DatasetDict
import numpy as np
import re
from sklearn.metrics import precision_score, f1_score, recall_score, accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from contractions import fix
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map 
import random

# Login to Weights & Biases (W&B) - Required in Kaggle
wandb.login()  # reads WANDB_API_KEY from the environment  # Replace with your actual API key
wandb.init(project="Spring2025", reinit=True)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dhruvjoshi892000 (dhruvjoshi892000-pace-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
# Ensure GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} for training")

Using cuda for training


In [4]:
import spacy

# Load spaCy model with GPU support
spacy.require_gpu()  # Ensure spaCy runs on GPU
nlp = spacy.load("en_core_web_sm")

# Verify if GPU is used
print("spaCy is using GPU:", spacy.prefer_gpu())


# Load spaCy's English NLP model
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Preprocess legal text for analysis using spaCy.
    Steps:
    - Lowercasing
    - Removing HTML tags
    - Removing URLs
    - Expanding contractions
    - Removing special characters
    - Removing numbers
    - Removing stop words
    - Lemmatization using spaCy
    - Removing extra whitespace
    """
    # Lowercase the text
    text = text.lower()
    
    # Remove HTML tags and URLs
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Expand contractions (optional)
    try:
        text = fix(text)
    except ImportError:
        pass  # Skip if the 'contractions' package is not available
    
    # Remove special characters and numbers (excluding spaces)
    text = re.sub(r'[^a-z\s]', '', text)

    # Process text using spaCy
    doc = nlp(text)
    
    # Lemmatize and remove stopwords
    text = ' '.join([token.lemma_ for token in doc if not token.is_stop])

    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text


# Define tokenization function
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)

# Load dataset from CSV
df = pd.read_csv("/kaggle/input/labels-web-of-law/15_labels_data.csv")  # Adjust file path if needed

# Apply preprocessing
# df = df[:100]
# df["text"] = df["text"].apply(preprocess_text)
processed_texts = [preprocess_text(text) for text in tqdm(df['text'], desc="Preprocessing texts")]

# Convert DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# Split dataset into train and validation
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset['train']
eval_dataset = dataset['test']


# Load tokenizer
model_name = "bert-base-uncased"  # Modify as needed
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
eval_dataset = eval_dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.10/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.7.1) was trained with spaCy v3.7.2 and may not be 100% compatible with the current version (3.8.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


spaCy is using GPU: True


Preprocessing texts: 100%|██████████| 8417/8417 [34:55<00:00,  4.02it/s]  


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7575 [00:00<?, ? examples/s]

Map:   0%|          | 0/842 [00:00<?, ? examples/s]

In [5]:
# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

In [11]:
# Load pre-trained model
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=15).to(device)

# Adjusted LoRA Configuration for Long-Form Text Classification
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS, 
    r=16,
    lora_alpha=32,
    lora_dropout=0.6,
    target_modules=[
        "attention.self.query",
        "attention.self.key",
        "attention.self.value",
        "attention.output.dense",
        "intermediate.dense",
        "output.dense"
    ],
    bias="none",
    inference_mode=False
)

# Integrate LoRA with the model
model = get_peft_model(model, lora_config).to(device)
model.print_trainable_parameters()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,665,743 || all params: 112,159,518 || trainable%: 2.3767


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    accuracy = accuracy_score(labels, predictions)
    balanced_acc = balanced_accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall = recall_score(labels, predictions, average='weighted', zero_division=0)
    f1 = f1_score(labels, predictions, average='weighted', zero_division=0)
    return {
        "accuracy": accuracy,
        "balanced_accuracy": balanced_acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


In [13]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/results",
#     evaluation_strategy="epoch",
#     save_strategy="epoch",
#     logging_dir="/kaggle/working/logs",
#     label_names=["labels"],
#     logging_steps=10,
#     num_train_epochs=20,  # More epochs for better legal text learning
#     per_device_train_batch_size=4,  # Lower batch size for stability
#     per_device_eval_batch_size=4,
#     learning_rate=2e-5,  # Reduce LR for better fine-tuning
#     warmup_ratio=0.1,  # Stabilize early training
#     weight_decay=0.01,
#     metric_for_best_model="balanced_accuracy",
#     load_best_model_at_end=True,
#     report_to=["wandb"],  # Enable W&B logging
#     fp16=torch.cuda.is_available(),  # Use FP16 if GPU is available
# )

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",  # Evaluate after every epoch
    save_strategy="epoch",        # Save the model every epoch
    learning_rate=2e-5,           # Reduced LR for stability
    per_device_train_batch_size=4,  # Increase if memory allows
    per_device_eval_batch_size=4,
    num_train_epochs=15,         # Keep training epochs
    weight_decay=0.05,           # Regularization to prevent overfitting
    logging_dir="./logs",
    logging_steps=50,            # Log progress every 50 steps
    load_best_model_at_end=True, # Load the best checkpoint
    metric_for_best_model="eval_loss",  # Select the best model based on validation loss
    lr_scheduler_type="cosine",  # Cosine decay learning rate
    warmup_ratio=0.1,            # 10% warmup steps
    gradient_accumulation_steps=2,  # Stabilizes training for smaller batch sizes
    fp16=True,                   # Mixed precision training
    report_to="wandb",           # Log results to Weights & Biases
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [14]:
# 🔧 Set Random Seed for Reproducibility
random.seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

In [15]:
# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

# Start training
trainer.train()

# Evaluate the model
results = trainer.evaluate()
print(results)


No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1
1,2.153600,2.109768,0.248219,0.076023,0.187847,0.248219,0.114793
2,1.732800,1.608157,0.515439,0.184610,0.443414,0.515439,0.419369
3,1.407600,1.359631,0.631829,0.301559,0.550641,0.631829,0.561107
4,1.269900,1.208779,0.652019,0.325072,0.542861,0.652019,0.586244
5,1.205300,1.134737,0.662708,0.350989,0.593045,0.662708,0.601758
6,1.050400,1.080182,0.673397,0.359143,0.600762,0.673397,0.626324
7,1.022500,1.036059,0.693587,0.393794,0.616821,0.693587,0.646582
8,0.973400,1.010871,0.695962,0.402160,0.613237,0.695962,0.646283
9,0.916700,0.996081,0.697150,0.398674,0.615620,0.697150,0.647234
10,0.865700,0.963274,0.710214,0.445096,0.662049,0.710214,0.668035


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

{'eval_loss': 0.9393097162246704, 'eval_accuracy': 0.7185273159144893, 'eval_balanced_accuracy': 0.47284286520820934, 'eval_precision': 0.6693805192234358, 'eval_recall': 0.7185273159144893, 'eval_f1': 0.6790898381280173, 'eval_runtime': 21.8721, 'eval_samples_per_second': 38.497, 'eval_steps_per_second': 4.846, 'epoch': 14.969376979936643}
